# UK grocery: cheapest effective price across five retailers

Runs one search across Tesco, Sainsbury's, Asda, Aldi and Morrisons with
[UK Grocery Prices](https://apify.com/yappman/uk-supermarket-price-scraper),
groups the rows into products by barcode (EAN) with `matchKey` as the fallback,
and reports the cheapest effective price for each product.

**Cost at Apify Free-tier prices:** at most **$0.255** per run
(100 product rows at $0.0025, plus one Actor Start at $0.005).
`maxItems` caps the rows and `max_total_charge_usd` caps the bill.

More data and the weekly index: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
from decimal import Decimal
import time

from pathlib import Path

import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)  # git-ignored: nothing you collect is committed


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped twice: maxItems in run_input limits the rows saved, and
    max_total_charge_usd stops the run if its charges would pass the cap.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
    )
    if run is None or run.status != "SUCCEEDED":
        raise RuntimeError(f"{actor_id} did not succeed: {None if run is None else run.status}")
    print(f"{actor_id}: run {run.id}")
    # Charges settle a few seconds after the run ends, so read the record again before printing them.
    time.sleep(5)
    settled = client.run(run.id).get()
    print("Charged events:", settled.charged_event_counts)
    return run

## 1. Run the search

`maxItems` is split equally between the shops that run, so 100 across five
shops gives up to 20 rows each. Branded products give the best barcode matches,
because own-label lines differ between retailers.

In [ ]:
ACTOR = "yappman/uk-supermarket-price-scraper"
RETAILERS = ["tesco", "sainsburys", "asda", "aldi", "morrisons"]
QUERIES = ["heinz baked beans", "pg tips tea bags", "lurpak spreadable"]
MAX_ITEMS = 100

# Free tier: 100 rows x $0.0025 + 1 start x $0.005 = $0.255
run = run_actor(
    ACTOR,
    {"mode": "search", "retailers": RETAILERS, "queries": QUERIES, "maxItems": MAX_ITEMS},
    max_charge_usd=0.26,
)
rows = list(client.dataset(run.default_dataset_id).iterate_items())
df = pd.DataFrame(rows)
print(len(df), "rows")
df.groupby("retailer").size()

## 2. Effective price

The effective price is the lower of the shelf price and the loyalty-card price
(Clubcard, Nectar, More Card). Multi-buy offers in `promotionText` are not
modelled. Rows reported out of stock are kept but are not eligible to be the
cheapest.

In [ ]:
df = df[df["price"].notna()].copy()
df["effectivePrice"] = df[["price", "loyaltyPrice"]].min(axis=1, skipna=True)
df["priceBasis"] = (df["loyaltyPrice"].notna() & (df["loyaltyPrice"] < df["price"])).map(
    {True: "loyalty", False: "shelf"}
)
df[["retailer", "name", "packSize", "price", "loyaltyPrice", "effectivePrice", "priceBasis"]].head(10)

## 3. Group rows into products: EAN first, `matchKey` as the fallback

* A barcode is usable when it has 8 to 14 digits, a valid GS1 check digit,
  at least 8 significant digits, and is not an in-store code (prefix 2 or 02).
  This drops retailer-internal IDs padded with zeros.
* Rows that share a usable barcode are the same product.
* A row with no usable barcode joins a barcode group only when exactly one
  group carries the same `matchKey`. Otherwise it groups with other
  barcode-less rows on `matchKey`.
* Two rows whose barcodes differ are never merged by `matchKey`.

`matchKey` is a normalised name and pack size. Its format changes no earlier
than 4 November 2026, so do not compare keys saved before and after that date.

In [ ]:
def usable_gtin(code) -> str | None:
    s = str(code).strip()
    if not s.isdigit() or not 8 <= len(s) <= 14:
        return None
    if len(s.lstrip("0")) < 8:  # padded internal id, not a real barcode
        return None
    s14 = s.zfill(14)
    if s14[1] == "2" or s14[1:3] == "02":  # GS1 restricted circulation
        return None
    digits = [int(c) for c in s14]
    total = sum(d * (3 if i % 2 == 0 else 1) for i, d in enumerate(digits[:-1]))
    return s14 if (10 - total % 10) % 10 == digits[-1] else None


parent: dict[str, str] = {}


def find(x: str) -> str:
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a: str, b: str) -> None:
    parent[find(a)] = find(b)


df = df.reset_index(drop=True)
df["gtins"] = df["ean"].apply(
    lambda codes: sorted({g for g in map(usable_gtin, codes or []) if g})
)

# Pass 1: barcodes
for i, gtins in df["gtins"].items():
    find(f"row:{i}")
    for g in gtins:
        union(f"row:{i}", f"gtin:{g}")

# Which barcode group(s) carry each matchKey?
mk_groups: dict[str, set[str]] = {}
for i, row in df[df["gtins"].str.len() > 0].iterrows():
    if row["matchKey"]:
        mk_groups.setdefault(row["matchKey"], set()).add(find(f"row:{i}"))

# Pass 2: matchKey fallback for rows without a usable barcode
for i, row in df[df["gtins"].str.len() == 0].iterrows():
    mk = row["matchKey"]
    if not mk:
        continue
    groups = mk_groups.get(mk, set())
    if len(groups) == 1:
        union(f"row:{i}", next(iter(groups)))
    elif not groups:
        union(f"row:{i}", f"mk:{mk}")

df["productId"] = [find(f"row:{i}") for i in df.index]
df["hasGtin"] = df["gtins"].str.len() > 0
print(df["productId"].nunique(), "products from", len(df), "rows")

## 4. Cheapest effective price per product

In [ ]:
def summarise(g: pd.DataFrame) -> pd.Series:
    eligible = g[g["inStock"] != False]  # noqa: E712  (None means not reported)
    best = (eligible if len(eligible) else g).sort_values("effectivePrice").iloc[0]
    method = "ean" if g["hasGtin"].all() else ("matchKey" if not g["hasGtin"].any() else "ean+matchKey")
    return pd.Series({
        "product": min(g["name"], key=len),
        "packSize": best["packSize"],
        "retailers": g["retailer"].nunique(),
        "cheapestRetailer": best["retailer"],
        "cheapest": best["effectivePrice"],
        "priceBasis": best["priceBasis"],
        "dearest": g["effectivePrice"].max(),
        "matchedBy": method,
    })


products = df.groupby("productId").apply(summarise, include_groups=False)
products["saving"] = (products["dearest"] - products["cheapest"]).round(2)
matched = products[products["retailers"] >= 2].sort_values("saving", ascending=False)
print(len(matched), "products found at two or more retailers;",
      (products["retailers"] == 1).sum(), "found at one")
matched.head(20)

In [ ]:
# Effective price for each matched product at each retailer
pivot = (
    df[df["productId"].isin(matched.index)]
    .pivot_table(index="productId", columns="retailer", values="effectivePrice", aggfunc="min")
    .join(matched[["product", "packSize"]])
    .set_index(["product", "packSize"])
)
pivot.to_csv(OUTPUT / "uk_cheapest_by_retailer.csv")
pivot

## Notes

* Prices are what each retailer showed when the run happened. Asda stock is for
  one store; prices are national.
* Rows and prices belong to the Actor's data terms, not to this repository's
  MIT licence. The CSV above is written to `output/`, which git ignores.